# A first reliability analysis

Build a resistance-minus-load model, run FORM and check its result against an exact normal-distribution calculation.

**Before you start:** basic Python and the [v2 installation](../install.md). This example uses only PySTRA's core dependencies.

## Define the physical event

Let resistance $R\sim N(10,1^2)$ and load $S\sim N(5,1^2)$ be independent, in the same force unit. Failure occurs when $g(R,S)=R-S\leq0$. The constructor arguments below are the mean and **standard deviation**, not variance.

In [1]:
import numpy as np
from scipy.stats import norm
import pystra as ra

model = ra.StochasticModel()
model.add_variable(ra.Normal("R", 10.0, 1.0))
model.add_variable(ra.Normal("S", 5.0, 1.0))
response = ra.LimitState(lambda R, S: R - S)

## Run and inspect the analysis

The workflow is model → analysis → result. Check convergence before using the probability or design point.

In [2]:
analysis = ra.FORM(model=model, limit_state=response)
result = analysis.run()
assert result.converged, result.message
print(f"Reliability index: {result.beta:.4f}")
print(f"Failure probability: {result.failure_probability:.6g}")
print("Physical design point:", dict(zip(result.variable_names, result.design_point_x)))

Reliability index: 3.5355
Failure probability: 0.000203476
Physical design point: {'R': np.float64(7.500000000000835), 'S': np.float64(7.500000000001386)}


## Check the known answer

Here $R-S$ is normal with mean 5 and variance $1^2+1^2=2$. Its exact reliability index is $5/\sqrt{2}$ and its failure probability is $\Phi(-5/\sqrt{2})$. FORM is exact for this plane in independent normal coordinates.

In [3]:
exact_beta = (10.0 - 5.0) / np.sqrt(1.0**2 + 1.0**2)
exact_probability = norm.sf(exact_beta)
np.testing.assert_allclose(result.beta, exact_beta, rtol=1e-6)
np.testing.assert_allclose(result.failure_probability, exact_probability, rtol=1e-6)
print(f"Exact failure probability: {exact_probability:.6g}")

Exact failure probability: 0.000203476


## Interpretation

The failure probability is about 0.0002035 for the event described by these input distributions. It is not automatically an annual probability: that interpretation requires an annual event model. The physical design point is approximately $R=S=7.5$.

For nonlinear models, numerical convergence does not establish approximation accuracy. Continue with [comparing FORM, SORM and simulation](ex_intro.ipynb), [interpreting results](../guides/results.rst), the [reliability API](../api/reliability.rst) and [design-point theory](../theory/design_point_methods.rst).

**Continue:** [User guide](../guides/results.rst) · [API reference](../api/reliability.rst) · [Theory](../theory/design_point_methods.rst)